# Option C: Depth 스케일 정렬 개선

## 문제 정의

Phase 4 통합 평가에서 **Depth RMSE = 5.09m** 로 Phase 1 (0.16m)보다 훨씬 높았다.

### 왜 RMSE가 높은가?

DepthAnythingV2는 **relative disparity** 를 출력한다:
- `pred` ∝ 1/distance (값이 클수록 **가까운** 픽셀)
- CARLA GT는 미터 단위 **depth** (값이 클수록 **먼** 픽셀)

기존 코드:
```python
s, t = lstsq([pred, 1], gt)   # pred와 gt의 관계가 선형이라 가정 ← 틀림
aligned = s * pred + t
```

올바른 관계:
```
depth_metric = 1 / (s · pred_disp + t)
```

## 이번 노트북에서 구현할 4가지 방법

| 방법 | 설명 |
|------|------|
| Baseline | 기존 linear scale-shift (depth space) |
| M1: Disparity-space | `1/(s·pred+t)` — 수식적으로 올바른 정렬 |
| M2: Log-space | `exp(a·log(pred)+b)` — 로그 스케일 불변 정렬 |
| M3: Metric 모델 | DepthAnythingV2-Metric-Outdoor (정렬 없이 직접 미터 출력) |

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

DATA_DIR  = Path('../../phase4_carla/data_collection/carla_dataset')
IMG_DIR   = DATA_DIR / 'images'
DEP_DIR   = DATA_DIR / 'depth'

img_files = sorted(IMG_DIR.glob('*.jpg'))[:20]  # 20장으로 평가
print(f'평가 이미지: {len(img_files)}장')

## 1. 공통 유틸리티

In [ ]:
def compute_depth_metrics(pred_m: np.ndarray, gt_m: np.ndarray, max_depth: float = 50.0) -> dict:
    """RMSE / AbsRel / δ1 / SI-RMSE 계산"""
    valid = (gt_m > 0.5) & (gt_m < max_depth) & (pred_m > 0)
    p, g = pred_m[valid], gt_m[valid]
    rmse    = float(np.sqrt(np.mean((p - g) ** 2)))
    absrel  = float(np.mean(np.abs(p - g) / (g + 1e-8)))
    delta1  = float((np.maximum(p / (g + 1e-8), g / (p + 1e-8)) < 1.25).mean())
    lp = np.log(np.clip(p, 1e-8, None))
    lg = np.log(np.clip(g, 1e-8, None))
    d  = lp - lg
    si_rmse = float(np.sqrt(np.mean(d ** 2) - 0.5 * np.mean(d) ** 2))
    return {'rmse': rmse, 'absrel': absrel, 'delta1': delta1, 'si_rmse': si_rmse}

def mean_metrics(results: list) -> dict:
    keys = results[0].keys()
    return {k: float(np.mean([r[k] for r in results])) for k in keys}

def load_gt(img_path: Path) -> np.ndarray:
    return np.load(str(DEP_DIR / f'{img_path.stem}.npy')).astype(np.float32)

print('유틸리티 로드 완료')

## 2. DepthAnythingV2 Relative 모델 로드

In [ ]:
REL_MODEL_ID = 'depth-anything/Depth-Anything-V2-Small-hf'

print(f'Relative 모델 로드: {REL_MODEL_ID}')
rel_processor = AutoImageProcessor.from_pretrained(REL_MODEL_ID)
rel_model     = AutoModelForDepthEstimation.from_pretrained(REL_MODEL_ID).to(device).eval()
print('로드 완료')

def predict_relative(img_path: Path) -> np.ndarray:
    """상대 disparity 맵 반환 (클수록 가까운 픽셀)"""
    img_pil = Image.open(img_path).convert('RGB')
    inputs  = rel_processor(images=img_pil, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = rel_model(**inputs)
    depth = torch.nn.functional.interpolate(
        outputs.predicted_depth.unsqueeze(1),
        size=img_pil.size[::-1], mode='bilinear', align_corners=False
    ).squeeze().cpu().numpy()
    return depth.astype(np.float32)

## 3. 정렬 함수 4종 구현

### Baseline: Linear scale-shift (depth space)

기존 방식. pred와 gt가 선형 관계라 가정:
$$\hat{d} = s \cdot pred + t, \quad \min_{s,t} \| \hat{d} - gt \|^2$$

**문제점**: pred는 disparity ∝ 1/depth이므로 이 가정이 틀림.

In [ ]:
def align_linear(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """기존 방식: depth space linear 정렬 (baseline)"""
    p, g = pred[mask].flatten(), gt[mask].flatten()
    A = np.stack([p, np.ones_like(p)], axis=1)
    s, t = np.linalg.lstsq(A, g, rcond=None)[0]
    return (pred * s + t).astype(np.float32)

print('align_linear (baseline) 정의 완료')

### M1: Disparity-space 정렬 (수식적으로 올바른 방법)

DepthAnythingV2 출력 `pred`는 disparity(역수 관계)이므로:
$$depth = \frac{1}{s \cdot pred + t}$$

최소제곱 풀이:
$$\min_{s,t} \left\| \frac{1}{gt} - (s \cdot pred + t) \right\|^2$$

→ `1/gt`를 타겟으로 `[pred, 1]`에 대해 lstsq

In [ ]:
def align_disparity(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray,
                    max_depth: float = 50.0) -> np.ndarray:
    """
    Disparity-space 정렬 (핵심 수식).
    pred (relative disparity) → metric depth

    수식: depth = 1 / (s·pred + t)
    최소제곱: 1/gt = s·pred + t 를 풀기

    버그 수정: clip 하한 = 1/max_depth (= 0.02 @ 50m)
      이전 코드: clip(1e-4) → 역수 = 10,000m → RMSE 폭발
      수정 코드: clip(1/50)  → 역수 = 50m  → 정상 range 유지
    """
    p = pred[mask].flatten()
    g = gt[mask].flatten()

    inv_g = 1.0 / np.clip(g, 1e-4, None)

    A = np.stack([p, np.ones_like(p)], axis=1)
    s, t = np.linalg.lstsq(A, inv_g, rcond=None)[0]

    inv_pred = s * pred + t
    # max_depth 기반 clip: 1/50 = 0.02 이상이어야 depth ≤ 50m
    inv_pred = np.clip(inv_pred, 1.0 / max_depth, None)
    return (1.0 / inv_pred).astype(np.float32)

print('align_disparity (M1) 정의 완료')

### M2: Log-space 정렬 (수정)

pred가 **disparity** (클수록 가까움)이므로, 그냥 `log(pred)`를 쓰면 방향이 반대.
먼저 `1/pred`로 변환(depth 방향)한 뒤 log-space fitting:

$$\log\left(\frac{1}{pred}\right) \xrightarrow{\text{→}} \log(depth_{approx})$$
$$\log(gt) = a \cdot \log\left(\frac{1}{pred}\right) + b$$

장점: 스케일 불변, 가깝고 먼 영역 균형 있게 학습  
이전 버그: pred(disparity)를 그대로 log → 방향 반전으로 a가 음수 → exp 폭발

In [ ]:
def align_logspace(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray,
                   max_depth: float = 50.0) -> np.ndarray:
    """
    Log-space 정렬 (수정 버전).

    버그: pred(disparity)를 그대로 log → gt와 방향 반대 → a<0 → exp 폭발
    수정: 먼저 1/pred 변환 (depth 방향 맞춤) → log fitting

    log(gt) = a·log(1/pred) + b
    → aligned = clip(exp(a·log(1/pred) + b), 0.1, max_depth)
    """
    # pred(disparity) → depth 방향으로 invert
    depth_from_disp = 1.0 / np.clip(pred, 1e-8, None)

    p = np.clip(depth_from_disp[mask].flatten(), 1e-8, None)
    g = np.clip(gt[mask].flatten(), 1e-8, None)

    log_p = np.log(p)
    log_g = np.log(g)

    A = np.stack([log_p, np.ones_like(log_p)], axis=1)
    a, b = np.linalg.lstsq(A, log_g, rcond=None)[0]

    log_dp_all = np.log(np.clip(depth_from_disp, 1e-8, None))
    result = np.exp(a * log_dp_all + b)
    # 안전 범위 clip
    return np.clip(result, 0.1, max_depth).astype(np.float32)

print('align_logspace (M2) 정의 완료 — disparity invert 수정 적용')

## 4. Baseline + M1 + M2 평가

DepthAnythingV2-Small (relative) 모델로 3가지 정렬 방법 비교

In [ ]:
results_baseline = []
results_m1       = []
results_m2       = []

for img_f in img_files:
    gt   = load_gt(img_f)
    pred = predict_relative(img_f)

    mask = (gt > 0.5) & (gt < 50.0)

    # Baseline
    aligned_base = align_linear(pred, gt, mask)
    results_baseline.append(compute_depth_metrics(aligned_base, gt))

    # M1: Disparity-space
    aligned_m1 = align_disparity(pred, gt, mask)
    results_m1.append(compute_depth_metrics(aligned_m1, gt))

    # M2: Log-space
    aligned_m2 = align_logspace(pred, gt, mask)
    results_m2.append(compute_depth_metrics(aligned_m2, gt))

mean_base = mean_metrics(results_baseline)
mean_m1   = mean_metrics(results_m1)
mean_m2   = mean_metrics(results_m2)

print('DepthAnythingV2-Small (Relative) 정렬 방법 비교')
print('=' * 65)
print(f'{"방법":<25} {"RMSE↓":>8} {"AbsRel↓":>8} {"δ1↑":>8} {"SI-RMSE↓":>10}')
print('-' * 65)
for name, m in [('Baseline (Linear)', mean_base), ('M1: Disparity-space', mean_m1), ('M2: Log-space', mean_m2)]:
    print(f'{name:<25} {m["rmse"]:>8.4f} {m["absrel"]:>8.4f} {m["delta1"]:>8.4f} {m["si_rmse"]:>10.4f}')

## 5. M3: DepthAnythingV2 Metric-Outdoor 모델

DepthAnythingV2에는 **metric depth를 직접 출력**하는 변형이 있다:
- `Depth-Anything-V2-Metric-Outdoor-Small-hf`: 야외(자율주행) 장면에 특화
- 스케일 정렬 없이 바로 미터 단위 depth 출력

ZoeDepth와 같은 역할 (metric supervision으로 학습됨)

In [ ]:
METRIC_MODEL_ID = 'depth-anything/Depth-Anything-V2-Metric-Outdoor-Small-hf'

print(f'Metric 모델 로드: {METRIC_MODEL_ID}')
metric_processor = AutoImageProcessor.from_pretrained(METRIC_MODEL_ID)
metric_model     = AutoModelForDepthEstimation.from_pretrained(METRIC_MODEL_ID).to(device).eval()
print('로드 완료')

def predict_metric(img_path: Path) -> np.ndarray:
    """직접 미터 단위 depth 출력 (정렬 불필요)"""
    img_pil = Image.open(img_path).convert('RGB')
    inputs  = metric_processor(images=img_pil, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = metric_model(**inputs)
    depth = torch.nn.functional.interpolate(
        outputs.predicted_depth.unsqueeze(1),
        size=img_pil.size[::-1], mode='bilinear', align_corners=False
    ).squeeze().cpu().numpy()
    return depth.astype(np.float32)

In [ ]:
results_m3 = []

for img_f in img_files:
    gt   = load_gt(img_f)
    pred = predict_metric(img_f)   # 이미 미터 단위
    results_m3.append(compute_depth_metrics(pred, gt))

mean_m3 = mean_metrics(results_m3)

print('M3: DepthAnythingV2-Metric-Outdoor 결과')
print(f'  RMSE    : {mean_m3["rmse"]:.4f} m')
print(f'  AbsRel  : {mean_m3["absrel"]:.4f}')
print(f'  δ1 acc  : {mean_m3["delta1"]:.4f}')
print(f'  SI-RMSE : {mean_m3["si_rmse"]:.4f}')

## 6. 전체 비교표

In [ ]:
methods = [
    ('Baseline\n(Linear, 기존)',      mean_base, '#d9534f'),
    ('M1: Disparity-space\n(수식 교정)', mean_m1, '#f0ad4e'),
    ('M2: Log-space\n(스케일 불변)',   mean_m2, '#5bc0de'),
    ('M3: Metric-Outdoor\n(정렬 없음)', mean_m3, '#5cb85c'),
]

print('\n전체 비교표')
print('=' * 75)
print(f'{"방법":<30} {"RMSE↓":>8} {"AbsRel↓":>9} {"δ1↑":>8} {"SI-RMSE↓":>10}')
print('-' * 75)
for name, m, _ in methods:
    n = name.replace('\n', ' ')
    print(f'{n:<30} {m["rmse"]:>8.4f} {m["absrel"]:>9.4f} {m["delta1"]:>8.4f} {m["si_rmse"]:>10.4f}')
print('=' * 75)
print(f'Phase4 원래 RMSE: 5.0936m (baseline 재현값과 비교)')

## 7. 시각화

### 7-1. RMSE / δ1 bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Depth 정렬 방법 비교', fontsize=14, fontweight='bold')

names  = [m[0].replace('\n', '\n') for m in methods]
colors = [m[2] for m in methods]
rmses  = [m[1]['rmse']   for m in methods]
d1s    = [m[1]['delta1'] for m in methods]

# RMSE (낮을수록 좋음)
bars = axes[0].bar(range(len(methods)), rmses, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('RMSE (m) ↓ 낮을수록 좋음', fontsize=12)
axes[0].set_xticks(range(len(methods)))
axes[0].set_xticklabels(names, fontsize=9)
axes[0].set_ylabel('RMSE (m)')
for bar, val in zip(bars, rmses):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# δ1 (높을수록 좋음)
bars2 = axes[1].bar(range(len(methods)), d1s, color=colors, edgecolor='black', linewidth=0.8)
axes[1].set_title('δ1 Accuracy ↑ 높을수록 좋음', fontsize=12)
axes[1].set_xticks(range(len(methods)))
axes[1].set_xticklabels(names, fontsize=9)
axes[1].set_ylabel('δ1')
axes[1].set_ylim(0, 1.05)
for bar, val in zip(bars2, d1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('depth_alignment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('차트 저장: depth_alignment_comparison.png')

### 7-2. 대표 이미지 시각화 (4방법 vs GT)

In [ ]:
# 대표 이미지 1장으로 시각화
sample_f = img_files[0]
gt_vis   = load_gt(sample_f)
pred_rel = predict_relative(sample_f)
pred_met = predict_metric(sample_f)

mask_vis = (gt_vis > 0.5) & (gt_vis < 50.0)
base_vis = align_linear(pred_rel, gt_vis, mask_vis)
m1_vis   = align_disparity(pred_rel, gt_vis, mask_vis)
m2_vis   = align_logspace(pred_rel, gt_vis, mask_vis)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(f'대표 이미지: {sample_f.name}', fontsize=13)

rgb = np.array(Image.open(sample_f).convert('RGB'))
vmin, vmax = 0, 50

axes[0, 0].imshow(rgb)
axes[0, 0].set_title('RGB 입력')
axes[0, 0].axis('off')

im = axes[0, 1].imshow(gt_vis, cmap='plasma', vmin=vmin, vmax=vmax)
axes[0, 1].set_title('GT Depth (CARLA)')
axes[0, 1].axis('off')
plt.colorbar(im, ax=axes[0, 1], fraction=0.046)

vis_pairs = [
    ('Baseline (Linear)', base_vis),
    ('M1: Disparity-space', m1_vis),
    ('M2: Log-space', m2_vis),
    ('M3: Metric-Outdoor', pred_met),
]

for ax, (title, dep) in zip(axes.flatten()[2:], vis_pairs):
    m = compute_depth_metrics(dep, gt_vis)
    im = ax.imshow(dep, cmap='plasma', vmin=vmin, vmax=vmax)
    ax.set_title(f'{title}\nRMSE={m["rmse"]:.3f}m  δ1={m["delta1"]:.3f}')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig('depth_visual_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('시각화 저장: depth_visual_comparison.png')

### 7-3. 오차 히트맵 (GT 대비 예측 오차)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('절대 오차 히트맵 |pred - GT| (m)', fontsize=13)

for ax, (title, dep) in zip(axes, vis_pairs):
    err = np.abs(dep - gt_vis)
    err[~mask_vis] = 0
    im = ax.imshow(err, cmap='hot', vmin=0, vmax=10)
    m = compute_depth_metrics(dep, gt_vis)
    ax.set_title(f'{title.split("(")[0].strip()}\nRMSE={m["rmse"]:.3f}m', fontsize=9)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig('depth_error_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('오차 히트맵 저장: depth_error_heatmap.png')

## 8. 분석 및 결론

### 핵심 원인 분석

In [ ]:
# 수식 관계 시각화: M1이 왜 작동하는지 직접 보여주기
sample_gt   = load_gt(img_files[0])
sample_pred = predict_relative(img_files[0])
mask_s = (sample_gt > 0.5) & (sample_gt < 50.0)

# 다운샘플 (시각화용)
gt_flat   = sample_gt[mask_s].flatten()[::50]
pred_flat = sample_pred[mask_s].flatten()[::50]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('DepthAnythingV2 출력과 GT의 관계 — M1 성공 원인 분석', fontsize=13)

# ── 왼쪽: pred vs GT (선형 공간) ──────────────────────────
axes[0].scatter(pred_flat, gt_flat, alpha=0.3, s=5, color='steelblue')
axes[0].set_xlabel('pred  (relative disparity)', fontsize=11)
axes[0].set_ylabel('GT depth (m)', fontsize=11)
axes[0].set_title('선형 공간: pred vs GT\n→ 쌍곡선 관계, 선형 정렬이 틀린 이유', fontsize=10)

# ── 오른쪽: pred vs 1/GT (M1이 실제 fitting하는 공간) ─────
# M1 수식: s·pred + t = 1/gt  → pred와 1/gt가 선형이어야 함
inv_gt = 1.0 / np.clip(gt_flat, 1e-4, None)
axes[1].scatter(pred_flat, inv_gt, alpha=0.3, s=5, color='tomato')
axes[1].set_xlabel('pred  (relative disparity)', fontsize=11)
axes[1].set_ylabel('1/GT  (inverse depth, m⁻¹)', fontsize=11)
axes[1].set_title('Disparity 공간: pred vs 1/GT\n→ 선형 관계 → M1 lstsq 적용 가능', fontsize=10)

# 회귀선 추가 (선형성 강조)
from numpy.polynomial import polynomial as P
c = np.polyfit(pred_flat, inv_gt, 1)
x_line = np.linspace(pred_flat.min(), pred_flat.max(), 100)
axes[1].plot(x_line, np.polyval(c, x_line), 'k--', linewidth=1.5,
             label=f'선형 회귀 (R²={np.corrcoef(pred_flat, inv_gt)[0,1]**2:.3f})')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('depth_relationship_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('분석 시각화 저장: depth_relationship_analysis.png')
print()
print('핵심: pred와 1/GT의 선형 관계가 M1(Disparity-space) 정렬의 수학적 근거')
print('      pred와 GT는 쌍곡선 관계 → Baseline이 실패한 이유')

## 9. 최종 결론 및 스킬업 라운드 2 전체 요약

In [ ]:
best_rmse   = min(mean_m1['rmse'], mean_m2['rmse'], mean_m3['rmse'])
best_delta1 = max(mean_m1['delta1'], mean_m2['delta1'], mean_m3['delta1'])
improvement = (mean_base['rmse'] - best_rmse) / mean_base['rmse'] * 100

print('=' * 60)
print('  Option C: Depth 스케일 정렬 개선 결과')
print('=' * 60)
print(f'  Baseline RMSE   : {mean_base["rmse"]:.4f} m')
print(f'  최고 RMSE (M1)  : {best_rmse:.4f} m')
print(f'  개선율          : {improvement:.1f}%')
print(f'  최고 δ1 acc     : {best_delta1:.4f}')
print()
print('  핵심 학습:')
print('  1. DepthAnythingV2 출력은 disparity (∝ 1/depth)')
print('     → depth space 선형 정렬은 수식이 틀림')
print('  2. 올바른 정렬: 1/gt를 target으로 lstsq → depth = 1/(s·pred+t)')
print('  3. clip 하한 = 1/max_depth (= 0.02 @ 50m) — 이 수치가 핵심')
print('  4. Metric 모델(M3)은 CARLA 도메인 갭으로 오히려 최악')
print('     → 합성 데이터에서는 정렬 기반 방식이 metric 모델보다 유리')
print()
print('=' * 60)
print('  스킬업 라운드 2 — 전체 결과')
print('=' * 60)
print()
print('  ┌─────────────────┬────────────────────────────────────┐')
print('  │   Option        │  개선 내용                          │')
print('  ├─────────────────┼────────────────────────────────────┤')
print('  │ A  Detection    │  mAP@50  0.43→0.68  (+58%)         │')
print('  │    Tracking     │  MOTA   -0.69→+0.29 (병목 해소)    │')
print('  ├─────────────────┼────────────────────────────────────┤')
print('  │ B  Segmentation │  mIoU   0.107→0.586 (+448%)        │')
print('  │                 │  SAM2 대비 5.5배 (태스크 미스매치)  │')
print('  ├─────────────────┼────────────────────────────────────┤')
print(f'  │ C  Depth        │  RMSE  {mean_base["rmse"]:.3f}→{best_rmse:.3f}m (-{improvement:.0f}%)         │')
print('  │                 │  disparity-space 수식 교정          │')
print('  └─────────────────┴────────────────────────────────────┘')